# 04 · Model Evaluation

**Purpose:** Calculate and evaluate predictive metrics (Brier, Logloss, and AUC) for the fitted models (M0–M3) on both training (in-sample) and test (out-of-sample) datasets.

**Inputs:**
- `reports/xfg_success/fg_full_with_predictions.csv` (from notebook 03)
- `reports/attempt_pi/attempt_pi_oof_predictions_final.csv` (from notebook 02)
- `models/m1/m1_fg_B_logit.rds`
- `data/fg_all.csv` (for era / season trend analyses)

**Outputs:**
- `reports/figures/metrics_table.csv` (Numeric metrics table)

**Requires:** notebooks 01–03 outputs.

In [1]:
# ============================================================
# 1. Parameters
# ============================================================
get_project_root <- function() {
  cwd <- getwd()
  if (basename(cwd) %in% c('notebooks', 'Reference', 'archive')) {
    return(dirname(cwd))
  } else if (basename(dirname(cwd)) == 'notebooks') {
    return(dirname(dirname(cwd)))
  } else {
    if (dir.exists(file.path(cwd, 'notebooks'))) {
      return(cwd)
    } else {
      return(sub('[/\\\\\\\\][^/\\\\\\\\]*$', '', cwd))
    }
  }
}
PROJECT_ROOT <- get_project_root()

data_dir          <- file.path(PROJECT_ROOT, 'data')
models_dir        <- file.path(PROJECT_ROOT, 'models')
final_models_dir  <- file.path(models_dir, 'final_models')
reports_dir       <- file.path(PROJECT_ROOT, 'reports')
figures_dir       <- file.path(reports_dir, 'figures')

# Evaluation window — must match notebook 03 to resolve the correct test ID file
EVAL_SEASON_MIN <- 2015L
EVAL_SEASON_MAX <- 2025L

# Canonical model artifact names from notebook 03
MODEL_FILE_M0     <- 'xfg_m0_dist_only_logit.rds'
MODEL_FILE_M1     <- 'xfg_m1_full_logit.rds'
MODEL_FILE_M2     <- 'xfg_m2_ipw_logit.rds'
MODEL_FILE_M2_POP <- 'xfg_m2_ipw_no_kicker_season_logit.rds'
MODEL_FILE_M3     <- 'xfg_m3_augmented_logit.rds'
MODEL_FILE_M3_NOPAT <- 'xfg_m3_augmented_nopat_logit.rds'

# Era split for the reviewer's Main-3 question ("you may find that you have a
# different answer to the selection bias question for earlier seasons versus
# late seasons"). 2020 is the break used throughout the paper's trend section.
ERA_BREAK <- 2020L

CALIB_BINS     <- 10L      # calibration bins
RESID_BINS     <- 10L      # residual curve bins
FIG_WIDTH      <- 8.0      # inches
FIG_HEIGHT     <- 5.5      # inches
FIG_DPI        <- 300
MIN_FGOE_ATT   <- 25L      # minimum FG attempts for leaderboard

# Model colour palette (Okabe-Ito inspired)
model_cols <- c('M0' = '#525252', 'M1' = '#0072B2', 'M2' = '#D55E00', 'M3' = '#009E73')

# Which model column to use as the primary "xFG" for FGOE etc.
XFG_COL    <- 'p_m2'       # IPW-corrected
AUG_COL    <- 'p_m3'       # augmented

if (!dir.exists(figures_dir)) dir.create(figures_dir, recursive = TRUE)
message('figures_dir: ', figures_dir)
message('final_models_dir: ', final_models_dir)


figures_dir: X:/My Files/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/figures



final_models_dir: X:/My Files/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/models/final_models



In [2]:
# ============================================================
# 2. Imports
# ============================================================
dependencies <- c(
  'dplyr', 'tibble', 'tidyr', 'readr', 'stringr', 'purrr',
  'ggplot2', 'ggrepel', 'scales', 'patchwork', 'pROC', 'glmmTMB'
)
installed <- rownames(installed.packages())
for (pkg in dependencies) {
  if (!pkg %in% installed) install.packages(pkg)
  suppressPackageStartupMessages(library(pkg, character.only = TRUE))
}

# Minimal paper theme: no grids, light border, clean typography
theme_paper <- function(base_size = 11) {
  ggplot2::theme_minimal(base_size = base_size) +
    ggplot2::theme(
      panel.grid.major = ggplot2::element_blank(),
      panel.grid.minor = ggplot2::element_blank(),
      panel.border = ggplot2::element_rect(fill = NA, colour = 'grey75', linewidth = 0.5),
      axis.line = ggplot2::element_line(colour = 'grey45', linewidth = 0.3),
      strip.background = ggplot2::element_rect(fill = 'grey95', colour = 'grey80', linewidth = 0.4),
      strip.text = ggplot2::element_text(face = 'bold', colour = 'grey25'),
      legend.position = 'bottom',
      legend.key = ggplot2::element_blank(),
      plot.title = ggplot2::element_text(face = 'bold', size = base_size + 2),
      plot.subtitle = ggplot2::element_text(colour = 'grey35', size = base_size),
      axis.title = ggplot2::element_text(size = base_size)
    )
}

save_fig <- function(p, fname, w = FIG_WIDTH, h = FIG_HEIGHT, dpi = FIG_DPI) {
  path <- file.path(figures_dir, fname)
  ggplot2::ggsave(path, plot = p, width = w, height = h, dpi = dpi)
  message('Saved: ', path)
  invisible(path)
}

message('Libraries loaded.')

Libraries loaded.



In [3]:
# ============================================================
# 3. Helper Functions
# ============================================================

brier   <- function(y, p) mean((y - p)^2, na.rm = TRUE)
logloss <- function(y, p, eps = 1e-15)
  -mean(y * log(pmin(pmax(p, eps), 1-eps)) + (1-y) * log(1-pmin(pmax(p, eps), 1-eps)), na.rm=TRUE)
auc_fn  <- function(y, p) tryCatch(as.numeric(pROC::auc(y, p, quiet = TRUE)), error = function(e) NA_real_)

calc_metrics <- function(y, p, set_name) {
  tibble::tibble(
    model = set_name, n = sum(!is.na(y) & !is.na(p)),
    brier = brier(y, p), logloss = logloss(y, p), auc = auc_fn(y, p)
  )
}

# Calibration data
calib_data <- function(y, p, nbins = CALIB_BINS, label = '') {
  df <- tibble::tibble(y = y, p = p) %>% filter(!is.na(y), !is.na(p))
  df$bin <- ggplot2::cut_number(df$p, nbins)
  df %>% group_by(bin) %>%
    summarise(obs = mean(y), pred = mean(p), n = n(), .groups = 'drop') %>%
    mutate(model = label)
}

## 4. Load Data

In [4]:
preds   <- readr::read_csv(file.path(reports_dir, 'xfg_success','fg_full_with_predictions.csv'),
                            show_col_types = FALSE)
preds_pi <- readr::read_csv(file.path(reports_dir, 'attempt_pi', 'attempt_pi_oof_predictions_final.csv'),
                             show_col_types = FALSE)
fg_all   <- readr::read_csv(file.path(data_dir, 'fg_all.csv'), show_col_types = FALSE)

# Load canonical models from models/final_models
m0 <- readRDS(file.path(final_models_dir, MODEL_FILE_M0))
m1 <- readRDS(file.path(final_models_dir, MODEL_FILE_M1))
m2 <- readRDS(file.path(final_models_dir, MODEL_FILE_M2))
m2_pop <- readRDS(file.path(final_models_dir, MODEL_FILE_M2_POP))
m3 <- readRDS(file.path(final_models_dir, MODEL_FILE_M3))
m3_nopat <- readRDS(file.path(final_models_dir, MODEL_FILE_M3_NOPAT))

message('Loaded canonical model artifacts from final_models.')

message('preds:    ', nrow(preds), ' FG rows')
message('preds_pi: ', nrow(preds_pi), ' rows | seasons ', min(preds_pi$season), '-', max(preds_pi$season))
message('fg_all:   ', nrow(fg_all), ' rows')

# Test set mask — filename mirrors notebook 03's TEST_GAME_IDS_FILE
test_game_ids_file <- file.path(data_dir, 'augmented',
  sprintf('test_fg_game_ids_%d_%d.csv', EVAL_SEASON_MIN, EVAL_SEASON_MAX))
test_game_ids <- if (file.exists(test_game_ids_file)) {
  readr::read_csv(test_game_ids_file, show_col_types = FALSE)$game_id
} else {
  warning('Test game IDs file not found: ', test_game_ids_file,
          '\nRun notebook 03 first to generate the split.')
  character(0)
}
message('Test game IDs loaded: ', length(test_game_ids), ' games from ', test_game_ids_file)

preds <- preds %>%
  mutate(
    split    = if_else(game_id %in% test_game_ids, 'test', 'train'),
    # Season-based era groupings (all data is 2015-2025)
    era      = dplyr::case_when(
      season <= 2018 ~ '2015-2018',
      season <= 2022 ~ '2019-2022',
      TRUE           ~ '2023-2025'
    ),
    fgoe_m1 = kick_made - p_m1,
    fgoe_m2 = kick_made - p_m2,
    fgoe_m3 = kick_made - p_m3
  )

# In-sample (all rows; 2015-2025 data only in this file)
preds_is <- preds
y_is     <- preds_is$kick_made

# Test subset
preds_test <- preds %>% filter(split == 'test')
y_test     <- preds_test$kick_made

message('in-sample rows: ', nrow(preds_is), ' | test rows: ', nrow(preds_test))
message('seasons: ', min(preds_is$season), '-', max(preds_is$season))

Loaded canonical model artifacts from final_models.



preds:    11548 FG rows



preds_pi: 22879 rows | seasons 2015-2025



fg_all:   137668 rows



Test game IDs loaded: 593 games from X:/My Files/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/data/augmented/test_fg_game_ids_2015_2025.csv



in-sample rows: 11548 | test rows: 2251



seasons: 2015-2025



## 4b. Population-Level xFG (M2 without kicker random effect)

For **player attribution** (FGOE leaderboards, skill scatter), we must not use a model that
already encodes individual kicker ability. M1/M2 include `(1 | kicker_player_id:season_f)`,
so their predictions absorb each kicker's skill, pushing FGOE toward zero for elite kickers
(e.g., Brandon Aubrey). The fix: refit M2 keeping only the **venue** random effect
`(1 | stadium_id)` — venue is environmental, not skill. This gives a population baseline
that a typical kicker of any given distance/environment is expected to make, making residuals
(`kick_made − p_m2_pop`) a clean measure of individual skill.

In [5]:
# ============================================================
# 4b. Population-level xFG from saved M2-pop model
# ============================================================
if (is.null(m2_pop)) stop('M2-pop model artifact not loaded from models/final_models')

# Align key types for prediction with random effects
preds_is <- preds_is %>%
  mutate(
    stadium_id = as.character(stadium_id),
    is_ot      = as.logical(is_ot)
  )

preds_test <- preds_test %>%
  mutate(
    stadium_id = as.character(stadium_id),
    is_ot      = as.logical(is_ot)
  )

# Use plain data.frames for glmmTMB prediction robustness
preds_is_df <- as.data.frame(preds_is)
preds_test_df <- as.data.frame(preds_test)

# Population-level predictions for fair player attribution
p_m2_pop_is <- predict(m2_pop, newdata = preds_is_df, type = 'response', allow.new.levels = TRUE)
p_m2_pop_test <- predict(m2_pop, newdata = preds_test_df, type = 'response', allow.new.levels = TRUE)

preds_is <- preds_is %>%
  mutate(
    p_m2_pop = p_m2_pop_is,
    fgoe_pop = kick_made - p_m2_pop
  )

preds_test <- preds_test %>%
  mutate(
    p_m2_pop = p_m2_pop_test,
    fgoe_pop = kick_made - p_m2_pop
  )

# Sanity checks
cat('\n=== M2-pop sanity checks ===\n')
cat('Mean fgoe_pop (should be ~0):', round(mean(preds_is$fgoe_pop, na.rm = TRUE), 4), '\n')
cat('Mean p_m2_pop:', round(mean(preds_is$p_m2_pop, na.rm = TRUE), 4),
    '  vs mean p_m2:', round(mean(preds_is$p_m2, na.rm = TRUE), 4), '\n')

aubrey <- preds_is %>%
  filter(stringr::str_detect(kicker_player_name, regex('aubrey', ignore_case = TRUE))) %>%
  summarise(
    n = n(),
    fgoe_total = sum(fgoe_pop, na.rm = TRUE),
    fgoe_per_att = mean(fgoe_pop, na.rm = TRUE),
    p_m2_pop_mean = mean(p_m2_pop, na.rm = TRUE)
  )

cat('\nAubrey (fgoe_pop): n=', aubrey$n,
    ' total=', round(aubrey$fgoe_total, 2),
    ' per_att=', round(aubrey$fgoe_per_att, 4), '\n')


=== M2-pop sanity checks ===


Mean fgoe_pop (should be ~0): 0.0036 


Mean p_m2_pop: 0.8591   vs mean p_m2: 0.8611 



Aubrey (fgoe_pop): n= 125  total= 9.82  per_att= 0.0786 


## 5. Metrics Table

In [6]:
# In-sample metrics (primary)
metrics_is <- dplyr::bind_rows(
  calc_metrics(y_is, preds_is$p_m0_full, 'M0 (distance only)'),
  calc_metrics(y_is, preds_is$p_m1_full, 'M1 (full GLMM)'),
  calc_metrics(y_is, preds_is$p_m2_full, 'M2 (IPW-corrected)'),
  calc_metrics(y_is, preds_is$p_m3_full, 'M3 (augmented)'),
  calc_metrics(y_is, preds_is$p_m3_nopat_full, 'M3 (no PAT)'),
  calc_metrics(y_is, preds_is$p_m2_pop_refit_full,  'M2-pop A (refit)'),
  calc_metrics(y_is, preds_is$p_m2_pop_zeroed_full, 'M2-pop B (kicker zeroed)')
) %>% mutate(split = 'in_sample', .before = 1)

# Test set metrics (supplemental)
metrics_test <- dplyr::bind_rows(
  calc_metrics(y_test, preds_test$p_m0, 'M0 (distance only)'),
  calc_metrics(y_test, preds_test$p_m1, 'M1 (full GLMM)'),
  calc_metrics(y_test, preds_test$p_m2, 'M2 (IPW-corrected)'),
  calc_metrics(y_test, preds_test$p_m3, 'M3 (augmented)'),
  calc_metrics(y_test, preds_test$p_m3_nopat, 'M3 (no PAT)'),
  calc_metrics(y_test, preds_test$p_m2_pop_refit,  'M2-pop A (refit)'),
  calc_metrics(y_test, preds_test$p_m2_pop_zeroed, 'M2-pop B (kicker zeroed)')
) %>% mutate(split = 'test', .before = 1)

metrics <- dplyr::bind_rows(metrics_is, metrics_test)
print(metrics)
readr::write_csv(metrics, file.path(figures_dir, 'metrics_table.csv'))


# A tibble: 14 × 6
   split     model                        n  brier logloss   auc
   <chr>     <chr>                    <int>  <dbl>   <dbl> <dbl>
 1 in_sample M0 (distance only)       11548 0.104    0.339 0.774
 2 in_sample M1 (full GLMM)           11548 0.100    0.326 0.804
 3 in_sample M2 (IPW-corrected)       11548 0.0983   0.322 0.806
 4 in_sample M3 (augmented)           11548 0.101    0.328 0.800
 5 in_sample M3 (no PAT)              11548 0.101    0.327 0.802
 6 in_sample M2-pop A (refit)         11548 0.103    0.336 0.780
 7 in_sample M2-pop B (kicker zeroed) 11548 0.104    0.337 0.780
 8 test      M0 (distance only)        2251 0.101    0.332 0.762
 9 test      M1 (full GLMM)            2251 0.100    0.330 0.769
10 test      M2 (IPW-corrected)        2251 0.103    0.337 0.761
11 test      M3 (augmented)            2251 0.101    0.333 0.766
12 test      M3 (no PAT)               2251 0.101    0.332 0.767
13 test      M2-pop A (refit)          2251 0.101    0.333 0.765
14 tes

In [7]:
# ============================================================
# 5b. Era-stratified selection-bias result (reviewer Main-3)
# ============================================================
# "In addition, you may find that you have a different answer to the selection
# bias question for earlier seasons versus late seasons."
#
# The selection-bias question is decided by M1 vs M2 on held-out data, so we
# re-answer it separately for pre-2020 and post-2020 test kicks. A positive
# delta means M2 is WORSE than M1 (higher Brier / log-loss).

era_label <- function(s) if_else(s < ERA_BREAK,
                                 paste0('pre-', ERA_BREAK), paste0('post-', ERA_BREAK))

era_oos <- preds_test %>%
  filter(!is.na(kick_made)) %>%
  mutate(era2 = era_label(season))

era_metrics <- era_oos %>%
  group_by(era2) %>%
  group_modify(~ dplyr::bind_rows(
    calc_metrics(.x$kick_made, .x$p_m0, 'M0 (distance only)'),
    calc_metrics(.x$kick_made, .x$p_m1, 'M1 (full GLMM)'),
    calc_metrics(.x$kick_made, .x$p_m2, 'M2 (IPW-corrected)'),
    calc_metrics(.x$kick_made, .x$p_m3, 'M3 (augmented)')
  )) %>%
  ungroup()

cat('\n=== OOS metrics by era (test kicks only) ===\n')
print(as.data.frame(era_metrics %>% mutate(across(where(is.numeric), ~round(., 4)))),
      row.names = FALSE)

# Head-to-head: does the M1-vs-M2 verdict flip across eras?
era_verdict <- era_metrics %>%
  filter(model %in% c('M1 (full GLMM)', 'M2 (IPW-corrected)')) %>%
  select(era2, model, n, brier, logloss, auc) %>%
  tidyr::pivot_wider(names_from = model, values_from = c(n, brier, logloss, auc)) %>%
  mutate(
    d_brier   = `brier_M2 (IPW-corrected)`   - `brier_M1 (full GLMM)`,
    d_logloss = `logloss_M2 (IPW-corrected)` - `logloss_M1 (full GLMM)`,
    d_auc     = `auc_M2 (IPW-corrected)`     - `auc_M1 (full GLMM)`,
    verdict   = if_else(d_brier < 0, 'M2 better', 'M1 better')
  )

cat('\n=== M1 vs M2 by era (delta = M2 - M1; positive means M2 is worse) ===\n')
print(as.data.frame(era_verdict %>%
        select(era2, `n_M1 (full GLMM)`, d_brier, d_logloss, d_auc, verdict) %>%
        mutate(across(where(is.numeric), ~round(., 5)))), row.names = FALSE)

readr::write_csv(era_metrics, file.path(figures_dir, 'metrics_by_era_oos.csv'))
readr::write_csv(era_verdict, file.path(figures_dir, 'm1_vs_m2_by_era_oos.csv'))
message('Saved: metrics_by_era_oos.csv, m1_vs_m2_by_era_oos.csv')



=== OOS metrics by era (test kicks only) ===


      era2              model    n  brier logloss    auc
 post-2020 M0 (distance only) 1275 0.0937  0.3113 0.7773
 post-2020     M1 (full GLMM) 1275 0.0928  0.3074 0.7833
 post-2020 M2 (IPW-corrected) 1275 0.0945  0.3131 0.7742
 post-2020     M3 (augmented) 1275 0.0943  0.3125 0.7831
  pre-2020 M0 (distance only)  976 0.1111  0.3597 0.7530
  pre-2020     M1 (full GLMM)  976 0.1105  0.3583 0.7588
  pre-2020 M2 (IPW-corrected)  976 0.1131  0.3675 0.7483
  pre-2020     M3 (augmented)  976 0.1108  0.3586 0.7553



=== M1 vs M2 by era (delta = M2 - M1; positive means M2 is worse) ===


      era2 n_M1 (full GLMM) d_brier d_logloss    d_auc   verdict
 post-2020             1275 0.00178   0.00568 -0.00913 M1 better
  pre-2020              976 0.00263   0.00914 -0.01045 M1 better


Saved: metrics_by_era_oos.csv, m1_vs_m2_by_era_oos.csv



## 5. Metrics Summary

In [8]:
# Print the final metrics summary table
print(metrics)

# A tibble: 14 × 6
   split     model                        n  brier logloss   auc
   <chr>     <chr>                    <int>  <dbl>   <dbl> <dbl>
 1 in_sample M0 (distance only)       11548 0.104    0.339 0.774
 2 in_sample M1 (full GLMM)           11548 0.100    0.326 0.804
 3 in_sample M2 (IPW-corrected)       11548 0.0983   0.322 0.806
 4 in_sample M3 (augmented)           11548 0.101    0.328 0.800
 5 in_sample M3 (no PAT)              11548 0.101    0.327 0.802
 6 in_sample M2-pop A (refit)         11548 0.103    0.336 0.780
 7 in_sample M2-pop B (kicker zeroed) 11548 0.104    0.337 0.780
 8 test      M0 (distance only)        2251 0.101    0.332 0.762
 9 test      M1 (full GLMM)            2251 0.100    0.330 0.769
10 test      M2 (IPW-corrected)        2251 0.103    0.337 0.761
11 test      M3 (augmented)            2251 0.101    0.333 0.766
12 test      M3 (no PAT)               2251 0.101    0.332 0.767
13 test      M2-pop A (refit)          2251 0.101    0.333 0.765
14 tes